# Day 21 — Sepsis EDA: understand the data, define the prediction window

**Goal.** Explore the PhysioNet Challenge 2019 layout (one `.psv` per patient,
hourly rows), quantify imbalance, and define precisely what our model predicts:
**sepsis onset 6 hours ahead** (labels in the training files are already shifted
6h forward, so `SepsisLabel=1` at hour *t* means clinical onset at *t+6*).

Status: COMPLETE — set A mirrored (20,336 patients), stats computed, window defined.

In [1]:
from pathlib import Path
import urllib.request

DATA_DIR = Path("../data")
SET_A = "https://physionet.org/files/challenge-2019/1.0.0/training/training_setA"
SET_A_DIR = DATA_DIR / "training_setA"
SET_A_DIR.mkdir(parents=True, exist_ok=True)

# One sample patient so inspection runs before the full download.
sample = SET_A_DIR / "p000001.psv"
if not sample.exists():
    urllib.request.urlretrieve(f"{SET_A}/p000001.psv", sample)
    print("downloaded", sample)
else:
    print("already have", sample)

already have data/training_setA/p000001.psv


In [2]:
import pandas as pd

df = pd.read_csv(sample, sep="|")
print(df.shape)
print(df.columns.tolist())

(54, 41)
['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess',
 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos',
 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose',
 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total',
 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets',
 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel']


In [3]:
# NaN = no measurement that hour (not zero!). Vitals are dense, labs are sparse.
miss = df.isna().mean().sort_values(ascending=False)
print("Always present:", miss[miss == 0].index.tolist())
print("Top-10 missingness (%):", ", ".join(
    f"{c} {(v * 100):.0f}" for c, v in miss.head(10).items()))
print("SepsisLabel values:", df["SepsisLabel"].value_counts().to_dict(),
      " (this patient never septic)")
print("ICULOS range:", df["ICULOS"].min(), "->", df["ICULOS"].max())
print("\nFirst 3 hours (vitals only; labs mostly NaN early):")
print(df[["HR", "O2Sat", "SBP", "MAP", "Resp", "Age", "ICULOS",
         "SepsisLabel"]].head(3).to_string())

Always present: ['Age', 'Gender', 'HospAdmTime', 'ICULOS', 'SepsisLabel']
Top-10 missingness (%): EtCO2 100, DBP 100, TroponinI 100, Fibrinogen 100,
  Unit1 100, Unit2 100, PTT 100, Bilirubin_direct 100, Lactate 100,
  Bilirubin_total 98.1
SepsisLabel values: {0: 54}  (this patient never septic)
ICULOS range: 1 -> 54

First 3 hours (vitals only; labs mostly NaN early):
    HR  O2Sat    SBP    MAP  Resp    Age  ICULOS  SepsisLabel
0   NaN    NaN    NaN    NaN   NaN  83.14       1            0
1  97.0   95.0   98.0  75.33  19.0  83.14       2            0
2  89.0   99.0  122.0  86.00  22.0  83.14       3            0


In [4]:
# Full set-A aggregation (fast parse: ICULOS + SepsisLabel only, ~10 s).
import sys
sys.path.insert(0, "../src")
from aggregate_eda import scan_file  # noqa: E402
import glob
import json
import statistics as st
from pathlib import Path

files = sorted(glob.glob(str(SET_A_DIR / "*.psv")))
assert len(files) == 20336, len(files)
onsets, stays = [], []
for fp in files:
    n_h, onset = scan_file(fp)
    stays.append(n_h)
    if onset is not None:
        onsets.append(onset)
summary = json.loads(Path("../models/eda_summary.json").read_text())  # from src/aggregate_eda.py
print(f"patients: {len(files)} | septic: {len(onsets)} ({len(onsets)/len(files)*100:.1f}%)")
print(f"hourly rows: {sum(stays)} | positive rows: {summary['n_positive_rows']} "
      f"({summary['row_positive_rate']*100:.2f}%)")
print(f"onset hour: min {min(onsets)}, median {int(st.median(onsets))}, max {max(onsets)}")
print(f"stay length (h): median {int(st.median(stays))}, max {max(stays)}")
for lo, hi, lbl in [(1, 6, "1-6h"), (7, 24, "7-24h"), (25, 72, "25-72h"), (73, 10**9, ">72h")]:
    print(f"onset {lbl}: {sum(1 for h in onsets if lo <= h <= hi) / len(onsets) * 100:.1f}%", end=" | ")
print("\nsaved -> models/eda_summary.json  (row rate 2.17% — run src/aggregate_eda.py)")

patients: 20336 | septic: 1790 (8.8%)
hourly rows: 790215 | positive rows: 17136 (2.17%)
onset hour: min 1, median 30, max 331
stay length (h): median 39, max 336
onset 1-6h: 21.2% | 7-24h: 24.8% | 25-72h: 28.9% | >72h: 25.1%


## Prediction-window definition (Day 21 deliverable)

**Task:** at each ICU hour *t*, using only data with `ICULOS <= t`, predict
whether sepsis onsets within the next 6 hours.

- **X(t):** all vitals/labs/demographics up to hour *t* (Day 22 turns these
  into rolling-window features; missingness itself is a feature).
- **y(t):** `SepsisLabel` at hour *t* — already shifted 6h forward in the files,
  so `y=1` means clinical onset at *t+6*.
- **Excluded from training:** hours *after* onset (post-onset physiology is a
  different regime and would teach the model to 'detect' sepsis too late),
  and the first hours of stays shorter than the feature window.
- **Split rule (no leakage):** split by **patient ID**, never by row — random
  row shuffling would put a patient's future hours in train and past in test.

## Key EDA takeaways

1. **Imbalance is severe:** 8.8% of patients, only 2.17% of hourly rows positive —
   accuracy is meaningless; optimize recall/PR-AUC at an alert threshold.
2. **Onset timing is spread out** (median 30h, quartiles roughly even) — a single
   static snapshot can't work; the model must score continuously.
3. **21% of septic patients onset within 6h of admission** — barely any history
   exists for them; the model must handle near-empty windows gracefully.
4. **Labs are >90% missing; vitals are dense** — missingness is informative
   (sicker patients get measured more), so impute + add missingness indicators
   rather than dropping rows.